In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import mysql.connector

conn = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST"),
    user=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    database=os.getenv("MYSQL_DATABASE"),
)

print("MySQL 연결 성공")

MySQL 연결 성공


In [3]:
cursor = conn.cursor()

cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table[0])

menu
sale_project
students
user_project


In [4]:
for table in ["user_Project", "sale_Project"]:

    print(f"\n[{table}]")

    cursor.execute(f"DESCRIBE `{table}`")

    for row in cursor.fetchall():
        print(row)


[user_Project]
('user_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('pw', 'varchar(255)', 'NO', '', None, '')
('name', 'varchar(50)', 'NO', '', None, '')
('gender', 'varchar(10)', 'YES', '', None, '')
('phone', 'varchar(20)', 'YES', '', None, '')
('region', 'varchar(50)', 'YES', '', None, '')
('note', 'varchar(255)', 'YES', '', None, '')

[sale_Project]
('sale_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('sale_date', 'date', 'NO', '', None, '')
('user_id', 'int', 'NO', 'MUL', None, '')
('product_name', 'varchar(100)', 'NO', '', None, '')
('unit_price', 'decimal(10,2)', 'NO', '', None, '')
('quantity', 'int', 'NO', '', None, '')


In [5]:
query = """
SELECT
    u.name,
    u.region,
    s.product_name,
    s.unit_price,
    s.quantity,
    s.unit_price * s.quantity AS total_price
FROM user_Project u
JOIN sale_Project s
    ON u.user_id = s.user_id
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('김철수', '서울', '키보드', Decimal('80000.00'), 1, Decimal('80000.00'))
('이영희', '경기', '모니터', Decimal('300000.00'), 1, Decimal('300000.00'))
('김철수', '서울', '마우스', Decimal('40000.00'), 2, Decimal('80000.00'))
('박민수', '서울', '키보드', Decimal('80000.00'), 2, Decimal('160000.00'))
('정우진', '경기', '헤드셋', Decimal('120000.00'), 1, Decimal('120000.00'))


In [6]:
MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE")

DATABASE_URL = (
    f"mysql+mysqlconnector://"
    f"{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}/{MYSQL_DATABASE}"
)

In [8]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(
    DATABASE_URL,
    include_tables=[
        "user_project",
        "sale_project"
    ],
    sample_rows_in_table_info=3,
)

In [10]:
print(db.dialect)
print(db.get_usable_table_names())
print(db.get_table_info())

mysql
['sale_project', 'user_project']

CREATE TABLE sale_project (
	sale_id INTEGER NOT NULL AUTO_INCREMENT, 
	sale_date DATE NOT NULL, 
	user_id INTEGER NOT NULL, 
	product_name VARCHAR(100) NOT NULL, 
	unit_price DECIMAL(10, 2) NOT NULL, 
	quantity INTEGER NOT NULL, 
	PRIMARY KEY (sale_id), 
	CONSTRAINT fk_sale_project_user FOREIGN KEY(user_id) REFERENCES user_project (user_id)
)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4

/*
3 rows from sale_project table:
sale_id	sale_date	user_id	product_name	unit_price	quantity
1	2026-08-01	1	키보드	80000.00	1
2	2026-08-03	2	모니터	300000.00	1
3	2026-08-05	1	마우스	40000.00	2
*/


CREATE TABLE user_project (
	user_id INTEGER NOT NULL AUTO_INCREMENT, 
	pw VARCHAR(255) NOT NULL, 
	name VARCHAR(50) NOT NULL, 
	gender VARCHAR(10), 
	phone VARCHAR(20), 
	region VARCHAR(50), 
	note VARCHAR(255), 
	PRIMARY KEY (user_id)
)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4

/*
3 rows from user_project table:
user_id	pw	name	gend

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
)

In [13]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm,
)

sql_tools = toolkit.get_tools()

In [14]:
for tool in sql_tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [15]:
SQL_SYSTEM_PROMPT = """
당신은 MySQL 데이터베이스 조회 전용 SQL Agent입니다.

사용 가능한 테이블은 다음 두 개입니다.

1. user_Project
2. sale_Project

규칙:

1. 사용자의 자연어 질문을 이해하고 필요한 SQL을 작성하세요.
2. 데이터 조회에는 SELECT만 사용하세요.
3. INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE는 절대 실행하지 마세요.
4. 필요한 경우 user_Project와 sale_Project를 user_id로 JOIN하세요.
5. SELECT * 사용은 가능한 피하고 필요한 컬럼만 조회하세요.
6. pw 컬럼은 어떠한 경우에도 조회하거나 사용자에게 보여주지 마세요.
7. SQL 실행 전에 쿼리가 올바른지 확인하세요.
8. 결과는 사용자가 이해하기 쉬운 한국어로 설명하세요.
9. 존재하지 않는 데이터는 추측하지 마세요.
"""

In [16]:
from langchain.agents import create_agent

sql_agent = create_agent(
    model=llm,
    tools=sql_tools,
    system_prompt=SQL_SYSTEM_PROMPT,
)

In [17]:
question = "서울에 거주하는 회원은 몇 명이야?"

result = sql_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(result["messages"][-1].content)

서울에 거주하는 회원은 총 2명입니다.


In [18]:
test_queries = [
    "서울에 거주하는 회원은 몇 명이야?",
    "VIP 회원의 이름과 지역을 알려줘.",
    "김철수가 구매한 상품을 알려줘.",
    "전체 판매 금액은 얼마야?",
    "가장 많이 팔린 상품은 뭐야?",
    "회원별 총 구매 금액을 알려줘.",
]

In [19]:
for question in test_queries:

    print("=" * 80)
    print("질문:", question)

    result = sql_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

    print(result["messages"][-1].content)

질문: 서울에 거주하는 회원은 몇 명이야?
서울에 거주하는 회원은 총 2명입니다.
질문: VIP 회원의 이름과 지역을 알려줘.
VIP 회원의 이름과 지역은 다음과 같습니다:

- 이영희: 경기
- 정우진: 경기

이 두 명의 회원이 VIP로 등록되어 있습니다.
질문: 김철수가 구매한 상품을 알려줘.
김철수가 구매한 상품은 "키보드"와 "마우스"입니다.
질문: 전체 판매 금액은 얼마야?
전체 판매 금액은 740,000원입니다.
질문: 가장 많이 팔린 상품은 뭐야?
가장 많이 팔린 상품은 "키보드"이며, 총 3개가 판매되었습니다.
질문: 회원별 총 구매 금액을 알려줘.
회원별 총 구매 금액은 다음과 같습니다:

- 김철수: 160,000원
- 이영희: 300,000원
- 박민수: 160,000원
- 정우진: 120,000원

각 회원의 구매 내역을 합산한 금액입니다.
